In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw/sources")

In [2]:
# POURQUOI : tous les seuils au même endroit. Si une règle change, on ne modifie
# QU'UNE cellule, et aucune fonction de traitement.
# (Hypothèses pédagogiques, issues de regles_metier.md et du cahier §6.2.)
SEUILS = {
    "vente_prix_min": 100_000,         # en dessous : prix atypique, à vérifier
    "vente_prix_max": 2_000_000_000,   # au-dessus : prix atypique, à vérifier
    "loyer_mensuel_min": 25_000,       # loyer par mois, en FCFA
    "loyer_mensuel_max": 5_000_000,
}

In [3]:
print(RAW.resolve())
print(RAW.exists())
sorted(p.name for p in RAW.iterdir())

/home/moussa-diaby/immo-analyse/data/raw/sources
True


['agence_teranga.xlsx',
 'import_historique.csv',
 'notaires.csv',
 'particuliers.json',
 'plateforme_immo_sn.csv']

In [4]:
plateforme = pd.read_csv(RAW / "plateforme_immo_sn.csv", sep=";", dtype=str)

print(plateforme.shape)
plateforme.head()

(875, 15)


,ref_plateforme,date_mise_en_ligne,type_transaction,categorie,region,commune,quartier,prix,devise,surface,pieces,sdb,description,statut,date_expiration
0,PLT-00001,2020-07-03,vente,Appartement,Tambacounda,Tambacounda,Nord,60581000,FCFA,173.3,1,1,Appartement à Tambacounda. Contact via platefo...,Vendu,2020-08-05
1,PLT-00002,2022-09-15,vente,Maison,Kolda,Kolda,Résidentiel,57.62 millions,FCFA,89.6,5,3,Maison à Kolda. Contact via plateforme.,Vendu,2022-11-19
2,PLT-00003,2019-01-15,vente,Terrain,Diourbel,Touba,Centre,9688000,FCFA,1457.3,0,0,Terrain à Touba. Contact via plateforme.,Vendu,2019-04-23
3,PLT-00004,2025-01-18,vente,Appartement,Thiès,Joal-Fadiouth,Sud,42273000,FCFA,69.8,3,2,Appartement à Joal-Fadiouth. Contact via plate...,En vente,2025-03-18
4,PLT-00005,2022-02-03,vente,Studio,Kaffrine,Kaffrine,Centre,26254000,FCFA,31.6,3,2,Studio à Kaffrine. Contact via plateforme.,Vendu,2022-03-26


Les valeurs manquantes
isna() ne détecte que les cases vraiment vides. Une case qui contient du texte comme "à négocier" joue le rôle d’un vide mais n’est pas comptée. On les cherchera à part, dans l’étude des formats.

In [5]:
# isna() renvoie True/False pour chaque case ; sum() compte les True par colonne
manquants = plateforme.isna().sum()

# Pourcentage = manquants / nombre total de lignes * 100, arrondi à 1 décimale
pourcentage = (manquants / len(plateforme) * 100).round(1)

# On assemble les deux dans un petit tableau, trié du plus incomplet au plus complet
resume = pd.DataFrame({"manquants": manquants, "pourcentage": pourcentage})
resume.sort_values("manquants", ascending=False)

,manquants,pourcentage
description,165,18.9
surface,5,0.6
ref_plateforme,0,0.0
categorie,0,0.0
region,0,0.0
date_mise_en_ligne,0,0.0
type_transaction,0,0.0
quartier,0,0.0
commune,0,0.0
devise,0,0.0


Les doublons

In [6]:
# duplicated() marque chaque ligne identique à une ligne PLUS HAUT dans le tableau
# (la première occurrence n'est pas comptée, seules les copies le sont)
doublons_exacts = plateforme.duplicated().sum()

# Même idée, mais uniquement sur la colonne identifiant
refs_dupliquees = plateforme["ref_plateforme"].duplicated().sum()

print("Lignes entièrement identiques :", doublons_exacts)
print("Références répétées           :", refs_dupliquees)

Lignes entièrement identiques : 15
Références répétées           : 15


Lignes concerrnées par les doublons

In [7]:
# keep=False marque TOUTES les lignes du groupe (l'original ET les copies),
# ce qui permet de les comparer côte à côte une fois triées par référence
plateforme[plateforme["ref_plateforme"].duplicated(keep=False)] \
    .sort_values("ref_plateforme") \
    .head(6)

,ref_plateforme,date_mise_en_ligne,type_transaction,categorie,region,commune,quartier,prix,devise,surface,pieces,sdb,description,statut,date_expiration
57,PLT-00058,2024-04-04,vente,Appartement,Diourbel,Mbacké,Sud,116544000,FCFA,160.2,5,2,Appartement à Mbacké. Contact via plateforme.,En vente,2024-06-27
864,PLT-00058,2024-04-04,vente,Appartement,Diourbel,Mbacké,Sud,116544000,FCFA,160.2,5,2,Appartement à Mbacké. Contact via plateforme.,En vente,2024-06-27
170,PLT-00171,2023-03-02,vente,Studio,Louga,Linguère,Populaire,41498000,FCFA,51.5,3,2,Studio à Linguère. Contact via plateforme.,Vendu,2023-06-18
869,PLT-00171,2023-03-02,vente,Studio,Louga,Linguère,Populaire,41498000,FCFA,51.5,3,2,Studio à Linguère. Contact via plateforme.,Vendu,2023-06-18
397,PLT-00398,2024-07-12,vente,Appartement,Fatick,Kaolack-Fatick,Résidentiel,31332000,FCFA,115.0,3,3,NaN,Vendu,2024-09-21
867,PLT-00398,2024-07-12,vente,Appartement,Fatick,Kaolack-Fatick,Résidentiel,31332000,FCFA,115.0,3,3,NaN,Vendu,2024-09-21


In [8]:
# POURQUOI : si une référence est répétée mais que la ligne n'est PAS identique,
# c'est un conflit (deux versions contradictoires), à traiter autrement qu'un doublon.
# On compare donc les deux comptages.

nb_refs_repetees = plateforme["ref_plateforme"].duplicated().sum()
nb_lignes_identiques = plateforme.duplicated().sum()

print("Références répétées  :", nb_refs_repetees)
print("Lignes identiques    :", nb_lignes_identiques)

# Si les deux nombres sont égaux, toute référence répétée est une copie exacte.
# S'ils diffèrent, la différence = nombre de conflits à examiner.
print("Conflits potentiels  :", nb_refs_repetees - nb_lignes_identiques)

Références répétées  : 15
Lignes identiques    : 15
Conflits potentiels  : 0


In [9]:
# Combien de ventes, de locations, de valeurs inattendues ?
# dropna=False : on compte aussi les cases vides au lieu de les cacher.
plateforme["type_transaction"].value_counts(dropna=False)
# La devise peut contenir une information en plus de la monnaie (une périodicité par exemple).
plateforme["devise"].value_counts(dropna=False)
# crosstab : une ligne par type de transaction, une colonne par devise.
# On voit si les deux colonnes sont liées.
pd.crosstab(plateforme["type_transaction"], plateforme["devise"], dropna=False)

devise,FCFA,FCFA/mois
type_transaction,,
location,0,177
vente,698,0


les formats du prix


In [10]:
def moule(serie: pd.Series) -> pd.Series:
    """Remplace chaque suite de chiffres par 9 et chaque mot par A, pour voir les formats."""
    return (
        serie.astype("string")                           # type texte qui accepte les valeurs manquantes
        .str.replace(r"\d+", "9", regex=True)            # une suite de chiffres -> un seul 9
        .str.replace(r"[A-Za-zÀ-ÿ]+", "A", regex=True)   # une suite de lettres (accents compris) -> un seul A
    )
# Combien de lignes pour chaque forme ? (dropna=False : on garde les valeurs manquantes)
moule(plateforme["prix"]).value_counts(dropna=False)

prix
9        804
9 9 9     50
9.9 A     15
A A        6
Name: count, dtype: Int64

Le « moule » remplace les chiffres par 9 et les mots par A, donc tous les prix écrits de la même façon ont le même moule, et on voit la liste des formats distincts.

In [11]:
# assign() ajoute une colonne "forme" sur une copie temporaire : plateforme n'est pas modifiée.
# groupby regroupe les lignes qui ont la même forme.
# size = nombre de lignes du groupe ; first = premier prix rencontré, pris comme exemple.
formes = (
    plateforme.assign(forme=moule(plateforme["prix"]))
    .groupby("forme", dropna=False)["prix"]
    .agg(nombre="size", exemple="first")
)
formes.sort_values("nombre", ascending=False)

,nombre,exemple
forme,,
9,804,60581000
9 9 9,50,438 746 000
9.9 A,15,57.62 millions
A A,6,à négocier


classer chaque famille
Forme	Classe	Raison	Règle
9	exploitable	uniquement des chiffres	conversion directe
9 9 9	normalisable	les espaces séparent les milliers, aucune autre lecture possible	enlever les espaces, puis convertir
9.9 A	normalisable	l’unité est écrite dans la valeur	nombre × 1 000 000
A A	inexploitable	aucun chiffre, on ne devine rien	quarantaine, motif « prix non numérique »

Le tableau n’affiche que la première valeur de chaque famille. Les 15 lignes 9.9 A contiennent-elles toutes « millions » ? Pourraient-elles contenir « milliards » ou « M » ? Les 6 A A sont-elles toutes « à négocier » ? On vérifie avant d’écrire une règle, sinon une règle sûre pour l’exemple peut être fausse pour le reste.

In [12]:
# POURQUOI : le groupby n'a montré qu'un exemple par famille. Ici on regarde TOUT le groupe.
formes_prix = moule(plateforme["prix"])

# Quels mots apparaissent dans la famille "9.9 A" ? extract(...) récupère la partie lettres.
mots = plateforme.loc[formes_prix == "9.9 A", "prix"].str.extract(r"([A-Za-zÀ-ÿ]+)")[0]
print(mots.value_counts())

# Quelles valeurs exactes dans la famille "A A" ?
print(plateforme.loc[formes_prix == "A A", "prix"].value_counts())

0
millions    15
Name: count, dtype: int64
prix
à négocier    6
Name: count, dtype: int64


c'est bon c'est confirmé

on va passer a l'étape des conversions 

le cahier impose l’indicateur « erreurs de conversion par champ ». Pour le mesurer, on tente la conversion sans rien corriger et on compte ce qui résiste

si on convertit directement prix sans rien corriger, quelles familles échouent, et combien de lignes en tout ?


le cahier insiste pour distinguer ventes et locations avant de normaliser un prix. Ici, on veut savoir si les familles « particulières » (espaces, millions, à négocier) se trouvent dans les ventes, les locations, ou les deux. Ça influence la façon dont on documentera les décisions.

In [13]:
# errors="coerce" : une valeur non convertible devient NaN au lieu de provoquer une erreur.
prix_num = pd.to_numeric(plateforme["prix"], errors="coerce")

# Échec = la case contenait quelque chose (notna) mais la conversion a donné NaN.
echecs = prix_num.isna() & plateforme["prix"].notna()

print("Échecs de conversion :", echecs.sum())
print("Conversions réussies :", prix_num.notna().sum())
# crosstab : une ligne par forme de prix, une colonne par type de transaction.
# Chaque case = nombre de lignes ayant cette forme ET ce type de transaction.
pd.crosstab(moule(plateforme["prix"]), plateforme["type_transaction"])

Échecs de conversion : 71
Conversions réussies : 804


type_transaction,location,vente
prix,,
9,177,627
9 9 9,0,50
9.9 A,0,15
A A,0,6


Le croisement apporte deux informations :

Constat	Conséquence
Les 177 locations sont toutes dans la famille 9	les loyers sont écrits proprement, aucune normalisation nécessaire
Les 71 prix problématiques sont tous des ventes

Bilan après normalisation prévue : sur 71 échecs, 65 sont corrigeables (50 espaces + 15 millions) et 6 restent inexploitables. On passe de 804 à 869 prix utilisables sur 875.

La fonction normaliser_prix

Pourquoi une fonction qui travaille sur UNE valeur : plus facile à comprendre et à tester. On l’appliquera ensuite à toute la colonne. Elle renvoie aussi un statut avec la valeur, car le rapport doit distinguer ce qui était conforme, ce qui a été corrigé et ce qui est rejeté (indicateurs obligatoires).

In [14]:
import re

def normaliser_prix(texte):
    """Convertit un prix écrit en texte en nombre (FCFA), sans jamais deviner.

    Renvoie un couple (valeur, statut) :
      (nombre, "conforme")     : déjà des chiffres seuls
      (nombre, "corrige")      : converti grâce à une règle sûre
      (None, "non_numerique")  : aucun nombre fiable -> quarantaine
      (None, "manquant")       : case vide
    """
    # Case vide : rien à convertir
    if texte is None or pd.isna(texte):
        return None, "manquant"

    brut = str(texte).strip()   # strip() enlève les espaces en début et fin

    # Règle 1 : chiffres seuls -> conversion directe
    if re.fullmatch(r"\d+", brut):
        return float(brut), "conforme"

    # Règle 2 : groupes de 3 chiffres séparés par des espaces ("438 746 000").
    # \s accepte aussi les espaces insécables, invisibles à l'œil mais fréquents dans les exports.
    if re.fullmatch(r"\d{1,3}(\s\d{3})+", brut):
        return float(re.sub(r"\s", "", brut)), "corrige"

    # Règle 3 : nombre décimal + le mot "millions" ("57.62 millions")
    m = re.fullmatch(r"(\d+(?:\.\d+)?)\s*millions?", brut, flags=re.IGNORECASE)
    if m:
        # round() : en informatique, 57.62 * 1_000_000 peut donner 57619999.99999999
        return float(round(float(m.group(1)) * 1_000_000)), "corrige"

    # Tout le reste ("à négocier", ...) : on ne devine rien
    return None, "non_numerique"

Test de la fonction avant de l’utiliser

Une fonction non testée est une supposition. On teste trois sortes de cas : normaux, invalides et limites.

In [15]:
# assert : si la condition est fausse, Python lève une erreur et montre laquelle.
# Cas normaux
assert normaliser_prix("60581000") == (60581000.0, "conforme")
assert normaliser_prix("438 746 000") == (438746000.0, "corrige")
assert normaliser_prix("57.62 millions") == (57620000.0, "corrige")

# Cas invalides
assert normaliser_prix("à négocier") == (None, "non_numerique")
assert normaliser_prix(None) == (None, "manquant")

# Cas limites : on ne doit PAS corriger ces valeurs ambiguës
assert normaliser_prix("12 34") == (None, "non_numerique")     # ce ne sont pas des milliers
assert normaliser_prix("1,250") == (None, "non_numerique")     # virgule : ambigu, on ne devine pas
assert normaliser_prix("-5000") == (None, "non_numerique")     # le signe moins n'est pas géré

# Un prix de 0 est bien ÉCRIT (chiffres seuls), donc la fonction l'accepte : conforme au FORMAT.
# Le juger absurde (un bien à 0 FCFA) est le rôle d'une autre étape : la plausibilité.
assert normaliser_prix("0") == (0.0, "conforme")

# Espace insécable (invisible à l'œil, fréquent dans les exports) : \s le reconnaît aussi
assert normaliser_prix("438\u00a0746\u00a0000") == (438746000.0, "corrige")

# Casse du mot : "Millions" doit marcher aussi, grâce à re.IGNORECASE
assert normaliser_prix("57.62 Millions") == (57620000.0, "corrige")

print("Tous les tests passent")

Tous les tests passent


### Application de normaliser_prix à la colonne prix

In [16]:
# apply() appelle normaliser_prix sur chaque valeur de la colonne "prix"
resultats = plateforme["prix"].apply(normaliser_prix)

# Chaque résultat est un couple (valeur, statut) : on le sépare en deux colonnes, dans un
# NOUVEAU tableau. plateforme (le brut) n'est pas modifiée.
prix_norm = pd.DataFrame(
    resultats.tolist(),
    columns=["prix_num", "prix_statut"],
    index=plateforme.index,   # même index que plateforme : chaque ligne reste alignée avec son origine
)

prix_norm["prix_statut"].value_counts()

prix_statut
conforme         804
corrige           65
non_numerique      6
Name: count, dtype: int64

les statistiques par type de transaction

Pourquoi : le minimum et le maximum montrent tout de suite si des valeurs sortent de l’ordinaire. La médiane (le « milieu » des valeurs) est plus fiable que la moyenne, parce qu’une seule valeur énorme tire la moyenne vers le haut sans changer la médiane.

In [17]:
# On rapproche le statut de transaction du prix normalisé, dans un tableau de travail.
# assign() crée une copie : ni plateforme ni prix_norm ne sont modifiés.
tableau = prix_norm.assign(type_transaction=plateforme["type_transaction"])

# Par défaut, Pandas affiche les grands nombres en notation scientifique (illisible).
# Ce réglage affiche des entiers avec séparateur de milliers, partout dans le notebook.
pd.set_option("display.float_format", "{:,.0f}".format)

# groupby : un bloc de statistiques par type de transaction.
# describe() donne : count, mean (moyenne), std, min, 25 %, 50 % (médiane), 75 %, max.
tableau.groupby("type_transaction")["prix_num"].describe()

,count,mean,std,min,25%,50%,75%,max
type_transaction,,,,,,,,
location,177,"397,288","290,689","40,000","190,000","315,000","545,000","1,480,000"
vente,692,"105,984,158","95,451,043","5,470,000","41,538,500","79,126,500","130,534,250","551,163,000"


### Seuils du prix (source : regles_metier.md et cahier des charges)
- **Bloquant (quarantaine)** : prix absent, non numérique ou <= 0.
- **Avertissement, vente** : prix < 100 000 ou > 2 000 000 000 FCFA.
- **Avertissement, location** : loyer mensuel < 25 000 ou > 5 000 000 FCFA.
- **Normalisable** : séparateurs de milliers, suffixe « millions » avec unité explicite.

Le premier comptage

In [18]:
# Un prix doit être strictement positif. On compte les violations.
# Les NaN ne sont pas comptés : comparer avec NaN donne False.
print("Prix <= 0 :", (tableau["prix_num"] <= 0).sum())

Prix <= 0 : 0


In [19]:
# Rétablit l'affichage normal des nombres décimaux
pd.reset_option("display.float_format")

In [20]:
# RAW pointe vers data/raw/sources.
# .parent remonte d'un niveau (data/raw), puis on descend dans le dossier referentiels.
ref_cat = pd.read_csv(RAW.parent / "referentiels" / "referentiel_categories.csv")
ref_cat

,categorie,synonymes,prix_min_indicatif,prix_max_indicatif,surface_min_typique,surface_max_typique
0,Appartement,appart|appartement|appt,25000000,120000000,45,180
1,Studio,studio,12000000,45000000,20,60
2,Chambre,chambre|chambre simple,5000000,18000000,10,30
3,Villa,villa,80000000,450000000,120,450
4,Maison,maison,35000000,180000000,60,250
5,Terrain,terrain|terrains,8000000,90000000,80,1500
6,Local commercial,local|local commercial|bureau|magasin,30000000,200000000,40,300


le premier comptage sur les ventes

In [21]:
# On isole les ventes : les seuils de vente ne valent pas pour les locations.
# (le tableau "tableau" a été créé à l'étape du describe : il doit déjà exister)
ventes = tableau[tableau["type_transaction"] == "vente"]
prix_v = ventes["prix_num"]

# Les NaN (les 6 "à négocier") ne sont comptés dans aucune ligne :
# comparer un NaN avec un nombre donne toujours False.
print("Ventes évaluables               :", prix_v.notna().sum())
print("Bloquant : prix <= 0            :", (prix_v <= 0).sum())
print("Avertissement : < seuil minimum :", ((prix_v > 0) & (prix_v < SEUILS["vente_prix_min"])).sum())
print("Avertissement : > seuil maximum :", (prix_v > SEUILS["vente_prix_max"]).sum())

Ventes évaluables               : 692
Bloquant : prix <= 0            : 0
Avertissement : < seuil minimum : 0
Avertissement : > seuil maximum : 0


Exploration  categorie

voir toutes les valeurs

In [22]:
# Combien de lignes pour chaque valeur écrite dans "categorie" ?
# dropna=False : on compte aussi les cases vides au lieu de les cacher.
plateforme["categorie"].value_counts(dropna=False)

categorie
Appartement         220
Maison              188
Villa               151
Terrain             123
Studio              107
Chambre              41
Local commercial     40
appart                5
Name: count, dtype: int64

repérer ce que le référentiel ne reconnaît pas tel quel

In [23]:
# POURQUOI : le référentiel donne les noms officiels (colonne "categorie") ET les synonymes
# admis (colonne "synonymes", séparés par "|"). On construit un dictionnaire
# "synonyme en minuscules" -> "nom officiel", directement depuis le fichier :
# les règles restent dans le référentiel, pas écrites en dur dans notre code.
synonymes = {}
for _, ligne in ref_cat.iterrows():                      # parcourt le référentiel ligne par ligne
    for syn in ligne["synonymes"].split("|"):            # "appart|appartement|appt" -> 3 synonymes
        synonymes[syn.strip().lower()] = ligne["categorie"]

# Pour chaque valeur de la plateforme, on teste : est-elle reconnue après
# un nettoyage minimal (espaces en trop retirés, minuscules) ?
valeurs = plateforme["categorie"].dropna().str.strip().str.lower()
reconnue = valeurs.isin(synonymes.keys())

print("Valeurs reconnues     :", reconnue.sum())
print("Valeurs NON reconnues :", (~reconnue).sum())

# Détail des valeurs écrites autrement que le nom officiel exact
plateforme.loc[~plateforme["categorie"].isin(ref_cat["categorie"]), "categorie"].value_counts(dropna=False)

Valeurs reconnues     : 875
Valeurs NON reconnues : 0


categorie
appart    5
Name: count, dtype: int64

la fonction normaliser_categorie

In [24]:
def normaliser_categorie(valeur, synonymes, officielles):
    """Ramène un type de bien à son nom officiel, sans jamais deviner.

    synonymes  : dictionnaire {synonyme en minuscules : nom officiel}, construit depuis le référentiel
    officielles: ensemble des 7 noms officiels
    Renvoie (nom officiel, statut) :
      (nom, "conforme")          : déjà écrit exactement comme le nom officiel
      (nom, "corrige")           : casse, espaces ou synonyme du référentiel
      (None, "hors_referentiel") : valeur inconnue -> on ne devine pas
      (None, "manquant")         : case vide
    """
    
    # Case vide : rien à normaliser
    if valeur is None or pd.isna(valeur):
        return None, "manquant"

    brut = str(valeur)

    # Déjà exact : aucune correction nécessaire
    if brut in officielles:
        return brut, "conforme"

    # Nettoyage minimal (espaces en trop, minuscules), puis recherche dans le référentiel
    cle = brut.strip().lower()
    if cle in synonymes:
        return synonymes[cle], "corrige"
    # Valeur absente du référentiel : on ne l'invente pas
    return None, "hors_referentiel"
    

In [25]:
officielles = set(ref_cat["categorie"])   # les 7 noms officiels

Les tests (cas normaux, invalides, limites)

In [26]:

# Cas normaux
assert normaliser_categorie("Appartement", synonymes, officielles) == ("Appartement", "conforme")
assert normaliser_categorie("appart", synonymes, officielles) == ("Appartement", "corrige")

# Cas limites : casse, espaces, synonymes à plusieurs mots
assert normaliser_categorie("  VILLA ", synonymes, officielles) == ("Villa", "corrige")
assert normaliser_categorie("chambre simple", synonymes, officielles) == ("Chambre", "corrige")
assert normaliser_categorie("Local", synonymes, officielles) == ("Local commercial", "corrige")

# Cas invalides : on ne devine rien
assert normaliser_categorie("Château", synonymes, officielles) == (None, "hors_referentiel")
assert normaliser_categorie("", synonymes, officielles) == (None, "hors_referentiel")
assert normaliser_categorie(None, synonymes, officielles) == (None, "manquant")

print("Tous les tests passent")

Tous les tests passent


In [27]:
resultats_cat = plateforme["categorie"].apply(
    lambda v: normaliser_categorie(v, synonymes, officielles)
)

categorie_norm = pd.DataFrame(
    resultats_cat.tolist(),
    columns=["categorie_norm", "categorie_statut"],
    index=plateforme.index,
)

categorie_norm["categorie_statut"].value_counts()

categorie_statut
conforme    870
corrige       5
Name: count, dtype: int64

comparaison les prix aux fourchettes par catégorie

rassembler les colonnes dans un tableau de travail

In [28]:
# concat(axis=1) colle les tableaux côte à côte. Ils ont tous le même index que plateforme,
# donc chaque ligne reste alignée avec son origine. plateforme n'est pas modifiée.
travail = pd.concat(
    [plateforme[["ref_plateforme", "type_transaction"]], prix_norm, categorie_norm],
    axis=1,
)
travail.head()

,ref_plateforme,type_transaction,prix_num,prix_statut,categorie_norm,categorie_statut
0,PLT-00001,vente,60581000.0,conforme,Appartement,conforme
1,PLT-00002,vente,57620000.0,corrige,Maison,conforme
2,PLT-00003,vente,9688000.0,conforme,Terrain,conforme
3,PLT-00004,vente,42273000.0,conforme,Appartement,conforme
4,PLT-00005,vente,26254000.0,conforme,Studio,conforme


répérage des ventes hors fourchette

In [29]:
# 1. On garde les ventes dont le prix est évaluable (les 6 "à négocier" sont exclues du dénominateur).
ventes = travail[(travail["type_transaction"] == "vente") & travail["prix_num"].notna()]

# 2. merge rapproche chaque vente de la fourchette de SA catégorie.
# how="left" : on garde toutes les ventes, même si une catégorie n'avait pas de fourchette.
ventes = ventes.merge(
    ref_cat[["categorie", "prix_min_indicatif", "prix_max_indicatif"]],
    left_on="categorie_norm",
    right_on="categorie",
    how="left",
)

# 3. Hors fourchette = en dessous du minimum OU (|) au-dessus du maximum de sa catégorie.
ventes["hors_fourchette"] = (
    (ventes["prix_num"] < ventes["prix_min_indicatif"])
    | (ventes["prix_num"] > ventes["prix_max_indicatif"])
)

# 4. Par catégorie : combien de ventes hors fourchette, sur combien évaluables.
ventes.groupby("categorie_norm")["hors_fourchette"].agg(hors_fourchette="sum", evaluables="count")

,hors_fourchette,evaluables
categorie_norm,,
Appartement,6,168
Chambre,1,33
Local commercial,2,34
Maison,4,153
Studio,4,84
Terrain,2,90
Villa,2,130


Dans le travail.head(), la ligne 1 (PLT-00002) montre 57620000.0 avec le statut corrige : c’est bien le « 57.62 millions » converti par ta fonction. Les trois tableaux (prix, catégorie, transaction) sont alignés ligne par ligne.

Dans le tableau par catégorie, les évaluables font 168 + 33 + 34 + 153 + 84 + 90 + 130 = 692, soit exactement tes ventes à prix valide. Aucune vente n’a été perdue par le merge.

Interprétation

Au total, 21 ventes sur 692 sont hors fourchette (3,0 %).

Catégorie	Hors fourchette	Évaluables	Taux
Local commercial	2	34	5,9 %
Studio	4	84	4,8 %
Appartement	6	168	3,6 %
Chambre	1	33	3,0 %
Maison	4	153	2,6 %
Terrain	2	90	2,2 %
Villa	2	130	1,5 %

Ce qu’on peut dire :

Le taux est faible et assez homogène (entre 1,5 % et 5,9 %). Aucune catégorie n’est massivement hors norme, donc il n’y a pas de problème systématique, comme un type de bien dont tous les prix seraient décalés.
Le plus haut taux (Local commercial, 5,9 %) repose sur 2 cas seulement. Sur 34 lignes, un cas de plus ou de moins change le taux de près de 3 points. Il ne faut pas en tirer de conclusion sur ce type de bien : l’échantillon est trop petit. Un taux n’a de sens que si on regarde aussi son dénominateur.
Ces fourchettes sont indicatives, pas des règles de validité. Sortir d’une fourchette ne veut pas dire que le prix est faux : une villa exceptionnelle peut dépasser 450 M.

Décision proposée : avertissement, pas quarantaine. Le cahier dit de signaler les valeurs hors fourchette du référentiel, pas de les supprimer. Les 21 ventes sont conservées avec un drapeau « hors fourchette catégorie ». La quarantaine est réservée aux règles bloquantes (prix absent, non numérique, ≤ 0).

### Regard des 21 Cas

In [30]:
# Contrôle : toutes les ventes ont-elles trouvé leur fourchette ? (doit afficher 0)
# Si une catégorie n'avait pas de fourchette, la comparaison donnerait False et la vente
# serait considérée "dans la fourchette" sans qu'on le voie.
print("Ventes sans fourchette :", ventes["prix_min_indicatif"].isna().sum())

# On isole les 21 ventes hors fourchette. copy() : on travaille sur une copie.
hors = ventes[ventes["hors_fourchette"]].copy()

# La borne dépassée : le minimum si le prix est en dessous, sinon le maximum.
# where(condition, autre) garde la valeur quand la condition est vraie, sinon prend "autre".
borne = hors["prix_min_indicatif"].where(
    hors["prix_num"] < hors["prix_min_indicatif"], hors["prix_max_indicatif"]
)

# Écart en % par rapport à cette borne : négatif = sous le minimum, positif = au-dessus du maximum.
# round(0) : on garde des entiers, lisibles avec le format d'affichage actuel.
hors["ecart_pct"] = ((hors["prix_num"] / borne - 1) * 100).round(0)

hors[["ref_plateforme", "categorie_norm", "prix_num",
      "prix_min_indicatif", "prix_max_indicatif", "ecart_pct"]].sort_values("ecart_pct")

Ventes sans fourchette : 0


,ref_plateforme,categorie_norm,prix_num,prix_min_indicatif,prix_max_indicatif,ecart_pct
274,PLT-00362,Appartement,17785000.0,25000000,120000000,-29.0
383,PLT-00501,Studio,45107000.0,12000000,45000000,0.0
493,PLT-00634,Studio,45453000.0,12000000,45000000,1.0
85,PLT-00115,Local commercial,203343000.0,30000000,200000000,2.0
259,PLT-00342,Appartement,125596000.0,25000000,120000000,5.0
230,PLT-00306,Studio,48860000.0,12000000,45000000,9.0
90,PLT-00120,Appartement,132373000.0,25000000,120000000,10.0
375,PLT-00493,Local commercial,230846000.0,30000000,200000000,15.0
351,PLT-00464,Maison,208177000.0,35000000,180000000,16.0
67,PLT-00092,Terrain,105826000.0,8000000,90000000,18.0
